# 📊 Exploratory Data Analysis — Customer Support Tickets
> **Task 2: Customer Support Ticket Triage & Draft Reply**  
> Dataset: `data/customer_support_tickets_200k.csv`  
> Date: 2026-03-05

---
### 📋 Analysis Outline
1. Setup & Load Data
2. Dataset Overview & Shape
3. Column Audit (relevance mapping)
4. Missing Values / Null Analysis
5. Category Distribution
6. Priority / Urgency Distribution
7. Channel Distribution
8. Language Distribution
9. Ticket Text Length Analysis
10. Time-based Analysis
11. Cross-Analysis (Category × Urgency)
12. Escalation & SLA Analysis
13. Customer Segment Analysis
14. Final Recommendations

## 🔧 1. Setup & Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ─── PATHS ───────────────────────────────────────────────────────────────────
BASE_DIR   = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_PATH  = os.path.join(BASE_DIR, 'data', 'customer_support_tickets_200k.csv')
PLOTS_DIR  = os.path.join(BASE_DIR, 'data', 'plots')
os.makedirs(PLOTS_DIR, exist_ok=True)

# ─── STYLE ───────────────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams.update({'figure.figsize': (12, 5), 'font.size': 12})

print(f'✅ Libraries loaded')
print(f'📂 Data path : {DATA_PATH}')
print(f'📂 Plots dir : {PLOTS_DIR}')

## 📥 2. Load Dataset

In [ ]:
df = pd.read_csv(DATA_PATH, encoding='utf-8', on_bad_lines='skip')

print(f'✅ Dataset loaded!')
print(f'   Rows    : {df.shape[0]:,}')
print(f'   Columns : {df.shape[1]}')
print()
print('📋 Columns:')
for i, col in enumerate(df.columns):
    print(f'   [{i:02d}] {col}')

## 👁️ 3. Dataset Overview

In [ ]:
# First 5 rows
df.head()

In [ ]:
# Dtypes and non-null counts
df.info()

In [ ]:
# Numeric statistics
df.describe().T

## 🗂️ 4. Column Audit — Relevance for Task 2

In [ ]:
relevance_map = {
    'ticket_id':                   ('🔑 Identifier',              'Keep as index'),
    'customer_name':               ('🗑️ Drop — PII',             'Remove'),
    'customer_email':              ('🗑️ Drop — PII',             'Remove'),
    'product':                     ('✅ Context',                 'Keep'),
    'category':                    ('🎯 TARGET — Classification', 'MUST KEEP'),
    'issue_description':           ('🎯 TARGET — Ticket text',    'MUST KEEP'),
    'resolution_notes':            ('📚 KB Source for RAG',       'MUST KEEP'),
    'priority':                    ('🎯 TARGET — Urgency',        'MUST KEEP → map to 3 levels'),
    'status':                      ('✅ Metadata',                'Keep'),
    'channel':                     ('✅ Metadata input',          'Keep'),
    'region':                      ('✅ Optional context',        'Keep'),
    'customer_age':                ('⬜ Optional',               'Drop or keep'),
    'customer_gender':             ('⬜ Optional',               'Drop or keep'),
    'subscription_type':           ('🚨 Urgency signal',         'Keep'),
    'customer_tenure_months':      ('⬜ Optional',               'Drop or keep'),
    'previous_tickets':            ('🚨 Urgency signal',         'Keep'),
    'customer_satisfaction_score': ('⬜ Eval metric',            'Keep for eval'),
    'first_response_time_hours':   ('⬜ SLA metric',             'Keep for eval'),
    'resolution_time_hours':       ('⬜ SLA metric',             'Keep for eval'),
    'ticket_created_date':         ('✅ Timestamp input',        'Keep'),
    'ticket_resolved_date':        ('⬜ Optional',               'Keep for eval'),
    'escalated':                   ('🚨 Urgency signal',         'Keep'),
    'sla_breached':                ('🚨 Urgency signal',         'Keep'),
    'operating_system':            ('⬜ Tech context',           'Keep for tech tickets'),
    'browser':                     ('⬜ Optional',               'Drop or keep'),
    'payment_method':              ('⬜ Billing context',        'Keep for billing tickets'),
    'language':                    ('✅ Multilingual input',     'Keep'),
    'preferred_contact_time':      ('⬜ Optional',               'Drop'),
    'issue_complexity_score':      ('🚨 Urgency signal',         'Keep'),
    'customer_segment':            ('🚨 Urgency signal',         'Keep'),
}

audit_df = pd.DataFrame.from_dict(relevance_map, orient='index', columns=['Role', 'Action'])
print('COLUMN RELEVANCE AUDIT (Task 2):')
display(audit_df)

In [ ]:
# Unique values per column
uniq_df = pd.DataFrame({
    'dtype':   df.dtypes,
    'nunique': df.nunique(),
    'sample':  [', '.join(df[c].dropna().astype(str).unique()[:3]) for c in df.columns]
})
display(uniq_df)

## ❓ 5. Missing Values / Null Analysis

In [ ]:
null_df = pd.DataFrame({
    'null_count': df.isnull().sum(),
    'null_pct':   (df.isnull().sum() / len(df) * 100).round(2)
})

cols_with_nulls = null_df[null_df['null_count'] > 0].sort_values('null_pct', ascending=False)

if cols_with_nulls.empty:
    print('✅ No missing values found! Dataset is complete.')
else:
    print(f'⚠️  {len(cols_with_nulls)} columns have missing values:')
    display(cols_with_nulls)

    fig, ax = plt.subplots(figsize=(10, max(4, len(cols_with_nulls)*0.5)))
    cols_with_nulls['null_pct'].plot(kind='barh', ax=ax, color='#e74c3c', edgecolor='white')
    ax.set_xlabel('Missing %', fontsize=12)
    ax.set_title('Missing Values by Column', fontsize=14, fontweight='bold')
    ax.axvline(5, color='orange', linestyle='--', linewidth=1.5, label='5% threshold')
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, '01_missing_values.png'), dpi=150, bbox_inches='tight')
    plt.show()

print('\n🔍 Empty string check for key text columns:')
for col in ['issue_description', 'resolution_notes', 'category', 'priority']:
    if col in df.columns:
        n = (df[col].astype(str).str.strip() == '').sum()
        print(f'   {col:<30}: {n:,} empty ({n/len(df)*100:.2f}%)')

## 🏷️ 6. Category Distribution

In [ ]:
cat_counts = df['category'].value_counts()
cat_df = pd.DataFrame({'count': cat_counts, 'pct': (cat_counts/len(df)*100).round(2)})
print(f'📊 Unique categories: {len(cat_counts)}')
display(cat_df)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Horizontal bar chart
colors = sns.color_palette('husl', len(cat_counts))
bars = ax1.barh(cat_counts.index[::-1], cat_counts.values[::-1], color=colors[::-1], edgecolor='white')
ax1.set_xlabel('Number of Tickets')
ax1.set_title('Ticket Category Distribution', fontsize=14, fontweight='bold')
for bar, val in zip(bars, cat_counts.values[::-1]):
    ax1.text(bar.get_width() + max(cat_counts)*0.01,
             bar.get_y() + bar.get_height()/2,
             f'{val:,} ({val/len(df)*100:.1f}%)', va='center', fontsize=9)

# Pie chart
wedges, texts, autotexts = ax2.pie(
    cat_counts.values, labels=cat_counts.index,
    autopct='%1.1f%%', colors=colors, startangle=90, pctdistance=0.82
)
ax2.set_title('Category Share (%)', fontsize=14, fontweight='bold')
for t in autotexts: t.set_fontsize(9)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '02_category_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\n📌 Task 2 requires: Refund | Login | Delivery | Billing | Account | Other')
print('   → Map/relabel dataset categories to these if they differ.')

## 🚨 7. Priority / Urgency Distribution

In [ ]:
pri_counts = df['priority'].value_counts()
print('📊 Original priority levels:')
print(pri_counts.to_string())

# Map to 3-level urgency
df['urgency'] = df['priority'].map({'Urgent': 'High', 'High': 'High', 'Medium': 'Medium', 'Low': 'Low'})
urg_counts = df['urgency'].value_counts()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

pal4 = {'Urgent': '#c0392b', 'High': '#e67e22', 'Medium': '#f1c40f', 'Low': '#27ae60'}
pal3 = {'High': '#e74c3c', 'Medium': '#f39c12', 'Low': '#27ae60'}

order4 = [p for p in ['Urgent','High','Medium','Low'] if p in pri_counts]
vals4 = [pri_counts[p] for p in order4]
bars4 = ax1.bar(order4, vals4, color=[pal4[p] for p in order4], edgecolor='white', width=0.6)
ax1.set_title('Original Priority (4-Level)', fontweight='bold')
ax1.set_ylabel('Count')
for b, v in zip(bars4, vals4):
    ax1.text(b.get_x()+b.get_width()/2, v+300, f'{v:,}\n({v/len(df)*100:.1f}%)',
             ha='center', va='bottom', fontsize=10)

order3 = [p for p in ['High','Medium','Low'] if p in urg_counts]
vals3  = [urg_counts[p] for p in order3]
bars3 = ax2.bar(order3, vals3, color=[pal3[p] for p in order3], edgecolor='white', width=0.6)
ax2.set_title('Mapped Urgency — Task 2 (3-Level)', fontweight='bold')
ax2.set_ylabel('Count')
for b, v in zip(bars3, vals3):
    ax2.text(b.get_x()+b.get_width()/2, v+300, f'{v:,}\n({v/len(df)*100:.1f}%)',
             ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '03_priority_urgency.png'), dpi=150, bbox_inches='tight')
plt.show()

## 📡 8. Channel Distribution

In [ ]:
ch_counts = df['channel'].value_counts()
print('📊 Channel counts:')
print(ch_counts.to_string())

fig, ax = plt.subplots(figsize=(10, 5))
colors = sns.color_palette('Set2', len(ch_counts))
bars = ax.bar(ch_counts.index, ch_counts.values, color=colors, edgecolor='white')
ax.set_title('Tickets by Channel', fontsize=14, fontweight='bold')
ax.set_ylabel('Count')
for b, v in zip(bars, ch_counts.values):
    ax.text(b.get_x()+b.get_width()/2, v+100, f'{v:,}\n({v/len(df)*100:.1f}%)',
            ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '04_channels.png'), dpi=150, bbox_inches='tight')
plt.show()

## 🌐 9. Language Distribution

In [ ]:
lang_counts = df['language'].value_counts()
print('📊 Language distribution:')
print(lang_counts.to_string())

fig, ax = plt.subplots(figsize=(12, 5))
colors = sns.color_palette('husl', len(lang_counts))
bars = ax.bar(lang_counts.index, lang_counts.values, color=colors, edgecolor='white')
ax.set_title('Tickets by Language', fontsize=14, fontweight='bold')
ax.set_ylabel('Count')
plt.xticks(rotation=30, ha='right')
for b, v in zip(bars, lang_counts.values):
    ax.text(b.get_x()+b.get_width()/2, v+100, f'{v:,}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '05_languages.png'), dpi=150, bbox_inches='tight')
plt.show()

eng = lang_counts.get('English', 0)
print(f'\n📌 English: {eng/len(df)*100:.1f}% | Multilingual: {(1-eng/len(df))*100:.1f}%')

## ✍️ 10. Ticket Text Length Analysis

In [ ]:
for col in ['issue_description', 'resolution_notes']:
    if col in df.columns:
        df[f'{col}_len'] = df[col].astype(str).str.len()
        s = df[f'{col}_len'].describe()
        print(f'\n📝 {col}:')
        print(f'   Min: {s["min"]:.0f}  |  Max: {s["max"]:.0f}  |  Mean: {s["mean"]:.0f}  |  Median: {s["50%"]:.0f} chars')

In [ ]:
text_len_cols  = [c for c in ['issue_description_len','resolution_notes_len'] if c in df.columns]
titles  = ['Issue Description Length (chars)', 'Resolution Notes Length (chars)']
colors2 = ['#3498db', '#2ecc71']

fig, axes = plt.subplots(1, len(text_len_cols), figsize=(14, 5))
if len(text_len_cols) == 1: axes = [axes]

for ax, col, title, color in zip(axes, text_len_cols, titles, colors2):
    data = df[col].clip(upper=df[col].quantile(0.99))
    ax.hist(data, bins=50, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(data.mean(),   color='red',    linestyle='--', label=f'Mean: {data.mean():.0f}')
    ax.axvline(data.median(), color='orange', linestyle='--', label=f'Median: {data.median():.0f}')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Characters')
    ax.set_ylabel('Frequency')
    ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '06_text_lengths.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Sample tickets per category
print('📝 SAMPLE ISSUE DESCRIPTIONS BY CATEGORY:')
print('='*80)
for cat in df['category'].unique():
    subset = df[df['category']==cat]['issue_description'].dropna()
    if len(subset) > 0:
        sample = str(subset.iloc[0])[:250]
        print(f'\n🏷️  {cat}')
        print(f'   {sample}...')

## 📅 11. Time-Based Analysis

In [ ]:
df['ticket_created_date'] = pd.to_datetime(df['ticket_created_date'], errors='coerce')
df['ticket_month'] = df['ticket_created_date'].dt.to_period('M')

print(f'📅 Date range: {df["ticket_created_date"].min().date()} → {df["ticket_created_date"].max().date()}')

tickets_per_month = df.groupby('ticket_month').size()

fig, ax = plt.subplots(figsize=(14, 5))
x = range(len(tickets_per_month))
ax.plot(x, tickets_per_month.values, color='#3498db', linewidth=2)
ax.fill_between(x, tickets_per_month.values, alpha=0.2, color='#3498db')
ax.set_xticks(x[::max(1,len(x)//12)])
ax.set_xticklabels([str(p) for p in tickets_per_month.index[::max(1,len(x)//12)]], rotation=45, ha='right')
ax.set_title('Tickets Created per Month', fontsize=14, fontweight='bold')
ax.set_ylabel('Ticket Count')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '07_tickets_over_time.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('⏱️ RESOLUTION TIME STATS:')
rt = df['resolution_time_hours'].dropna()
print(f'   Mean   : {rt.mean():.1f} hrs')
print(f'   Median : {rt.median():.1f} hrs')
print(f'   P95    : {rt.quantile(0.95):.1f} hrs')

print('\n⏱️ By Urgency:')
display(df.groupby('urgency')['resolution_time_hours'].agg(['mean','median','count']).round(1))

## 🔥 12. Cross-Analysis: Category × Urgency

In [ ]:
cross = pd.crosstab(df['category'], df['urgency'], normalize='index') * 100

col_order = [c for c in ['High','Medium','Low'] if c in cross.columns]
cross = cross[col_order]

fig, ax = plt.subplots(figsize=(10, max(5, len(cross)*0.5)))
sns.heatmap(cross, annot=True, fmt='.1f', cmap='RdYlGn_r',
            linewidths=0.5, cbar_kws={'label': '% within category'}, ax=ax)
ax.set_title('Category vs Urgency Heatmap (% within category)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '08_category_urgency_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Channel vs Category stacked bar
cross2 = pd.crosstab(df['channel'], df['category'])
cross2_pct = cross2.div(cross2.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(13, 5))
cross2_pct.plot(kind='bar', stacked=True, ax=ax, edgecolor='white', width=0.65)
ax.set_title('Category Distribution by Channel (%)', fontsize=13, fontweight='bold')
ax.set_ylabel('Percentage')
ax.set_xlabel('Channel')
ax.legend(title='Category', bbox_to_anchor=(1.01, 1), fontsize=9)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '09_channel_category.png'), dpi=150, bbox_inches='tight')
plt.show()

## 🚨 13. Escalation & SLA Analysis

In [ ]:
esc_counts = df['escalated'].value_counts()
sla_counts = df['sla_breached'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, counts, title in zip(axes,
    [esc_counts, sla_counts],
    ['Escalated?', 'SLA Breached?']):
    colors_pie = ['#e74c3c' if k=='Yes' else '#27ae60' for k in counts.index]
    ax.pie(counts.values, labels=counts.index, autopct='%1.1f%%',
           colors=colors_pie, startangle=90)
    ax.set_title(title, fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '10_escalation_sla.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\n🚨 ESCALATION RATE BY CATEGORY:')
esc_rate = df.groupby('category')['escalated'].apply(
    lambda x: (x=='Yes').mean()*100).round(2).sort_values(ascending=False)
display(esc_rate.rename('escalation_rate_%'))

In [ ]:
# Numeric correlation matrix
num_cols = [c for c in [
    'customer_age','customer_tenure_months','previous_tickets',
    'customer_satisfaction_score','first_response_time_hours',
    'resolution_time_hours','issue_complexity_score'
] if c in df.columns]

corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Numeric Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '11_correlation_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

## 👥 14. Customer Segment Analysis

In [ ]:
seg_counts = df['customer_segment'].value_counts()
sub_counts = df['subscription_type'].value_counts()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.pie(seg_counts.values, labels=seg_counts.index,
        autopct='%1.1f%%', colors=sns.color_palette('Set2', len(seg_counts)), startangle=90)
ax1.set_title('Customer Segment', fontsize=13, fontweight='bold')

ax2.pie(sub_counts.values, labels=sub_counts.index,
        autopct='%1.1f%%', colors=sns.color_palette('Set3', len(sub_counts)), startangle=90)
ax2.set_title('Subscription Type', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '12_segments.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\n📊 Avg issue_complexity_score by segment:')
display(df.groupby('customer_segment')['issue_complexity_score'].mean().round(2))

## ✅ 15. Final Recommendations & Column Selection

In [ ]:
print('='*70)
print('📋 FINAL COLUMN SELECTION FOR TASK 2')
print('='*70)

selection = {
    '🎯 Core (must keep)':       ['ticket_id','issue_description','category','priority','urgency','channel','ticket_created_date'],
    '📚 KB Source for RAG':      ['resolution_notes','product'],
    '🚨 Urgency Boost Features': ['escalated','sla_breached','issue_complexity_score','subscription_type','customer_segment','previous_tickets'],
    '✅ Useful Optional':        ['language','region','status','resolution_time_hours','operating_system','payment_method'],
    '🗑️ Drop (PII/noise)':       ['customer_name','customer_email','customer_age','customer_gender','preferred_contact_time','browser'],
}

for group, cols in selection.items():
    print(f'\n{group}:')
    for c in cols: print(f'   • {c}')

print('\n' + '='*70)
print('KEY DECISIONS:')
print('  1. Map priority → urgency: Urgent/High→High, Medium→Medium, Low→Low')
print('  2. Use resolution_notes → data/kb/ as RAG knowledge base source')
print('  3. Dataset is ~balanced across categories and urgency levels')
print('  4. Multilingual handling needed (~16-20% non-English tickets)')
print('  5. No significant missing values — minimal preprocessing needed')
print('='*70)

In [ ]:
# Save processed dataset with only required columns + urgency
core_cols = [
    'ticket_id','product','category','issue_description','resolution_notes',
    'priority','urgency','status','channel','region',
    'subscription_type','previous_tickets','escalated','sla_breached',
    'ticket_created_date','language','issue_complexity_score','customer_segment',
    'resolution_time_hours','first_response_time_hours'
]
core_cols = [c for c in core_cols if c in df.columns]

df_core = df[core_cols].copy()

out_path = os.path.join(BASE_DIR, 'data', 'tickets_processed.csv')
df_core.to_csv(out_path, index=False)

print(f'✅ Saved: data/tickets_processed.csv')
print(f'   {df_core.shape[0]:,} rows × {df_core.shape[1]} columns')
print(f'   Dropped : customer_name, customer_email, customer_gender, etc. (PII)')
print(f'\n🎉 EDA Complete! Next steps:')
print(f'   → 02_preprocessing.ipynb  : clean & tokenize text')
print(f'   → 03_classification.ipynb : train category + urgency classifier')
print(f'   → 04_kb_ingestion.ipynb   : extract resolution_notes → vector KB')
print(f'   → 05_rag_pipeline.ipynb   : RAG retrieval + grounded reply generation')